# Stock Analyst — MCP Jupyter Walkthrough

Five cells that build from raw MCP protocol to a full multi-turn agentic loop.

**Prerequisites**
1. MCP server running: `docker compose up -d mcp-server` (from this directory)
2. Environment variables set: `ANTHROPIC_API_KEY`, `MCP_SERVER_URL`
3. Dependencies installed: `pip install -r requirements.txt`

| Cell | What it demonstrates |
|------|----------------------|
| 1 | Imports and config |
| 2 | Connect to MCP server and list tool schemas |
| 3 | Raw tool call — **no Claude involved** |
| 4 | Single-turn Claude query with tool resolution |
| 5 | Multi-turn agentic loop (reusable function) |

## Cell 1 — Setup & imports

In [4]:
import asyncio
import json
import os

from anthropic import Anthropic
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.sse import sse_client

# ---------------------------------------------------------------------------
# Load .env from this notebook's directory (if present).
# ---------------------------------------------------------------------------
load_dotenv(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".env"))

# ---------------------------------------------------------------------------
# Config — set these in your environment or edit here directly.
# ---------------------------------------------------------------------------
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
MCP_SERVER_URL = os.environ.get("MCP_SERVER_URL", "http://localhost:8001/sse")

SUPPORTED_TICKERS = ["AAPL", "AMZN", "GOOGL", "META", "MSFT", "NFLX", "NVDA", "TSLA"]

SYSTEM_PROMPT = f"""You are a stock analyst assistant specialising in megacap technology stocks.

You have access to real-time and historical data for: {', '.join(SUPPORTED_TICKERS)}.
Use the available tools whenever you need data to answer a question accurately.

Guidelines:
- Format large numbers in billions/millions for readability (e.g. $2.8T, $192B).
- Always state the data period when discussing prices or financials.
- If a user asks about a ticker you don't support, politely say so and list the ones you do.
- Be concise but informative.
"""

assert ANTHROPIC_API_KEY, "Set ANTHROPIC_API_KEY before running cells 4–5"
print(f"MCP server: {MCP_SERVER_URL}")
print("Imports OK")

MCP server: http://localhost:8001/sse
Imports OK


## Cell 2 — Connect & list tools

Open an SSE connection, call `session.list_tools()`, and inspect the raw MCP tool schemas.
This is all MCP protocol — Claude is not involved.

In [5]:
async def list_tools():
    async with sse_client(MCP_SERVER_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.list_tools()

            print(f"{len(result.tools)} tools available:\n")
            for tool in result.tools:
                print(f"  {tool.name}")
                print(f"    {tool.description}")
                props = tool.inputSchema.get("properties", {})
                for param, schema in props.items():
                    desc = schema.get("description", "")
                    print(f"    param '{param}': {desc}")
                print()

await list_tools()

4 tools available:

  get_current_price
    
    Get the current price, daily change, volume, and market cap for a stock.
    ticker: one of AAPL, AMZN, GOOGL, META, MSFT, NFLX, NVDA, TSLA
    
    param 'ticker': 

  get_stock_overview
    
    Get company overview: name, sector, P/E ratio, 52-week range, dividend yield,
    and a short business description.
    ticker: one of AAPL, AMZN, GOOGL, META, MSFT, NFLX, NVDA, TSLA
    
    param 'ticker': 

  get_price_history
    
    Get historical daily OHLCV data for a stock.
    ticker: one of AAPL, AMZN, GOOGL, META, MSFT, NFLX, NVDA, TSLA
    period: one of 5d, 1mo, 3mo, 1y  (default: 1mo)
    Returns a list of {date, open, high, low, close, volume} records.
    
    param 'ticker': 
    param 'period': 

  get_financials
    
    Get annual revenue, net income, and trailing EPS for the last 4 fiscal years.
    ticker: one of AAPL, AMZN, GOOGL, META, MSFT, NFLX, NVDA, TSLA
    
    param 'ticker': 



## Cell 3 — Raw tool call (no Claude)

Call an MCP tool directly via `session.call_tool()`.  
Claude is **not** involved — this is a direct client ↔ server protocol call.

Key insight: **Claude is just one consumer of MCP**. Any code that can hold an
`mcp.ClientSession` can call tools. The AI layer is optional.

In [6]:
async def raw_tool_call(tool_name: str, args: dict):
    async with sse_client(MCP_SERVER_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, args)
            # result.content[0].text is raw JSON from the MCP server
            raw = result.content[0].text
            return json.loads(raw)

# --- Example 1: current price ---
price_data = await raw_tool_call("get_current_price", {"ticker": "AAPL"})
print("get_current_price(AAPL):")
print(json.dumps(price_data, indent=2))

print()

# --- Example 2: recent price history ---
history = await raw_tool_call("get_price_history", {"ticker": "NVDA", "period": "5d"})
print(f"get_price_history(NVDA, 5d) — {len(history['data'])} rows:")
for row in history["data"]:
    print(f"  {row['date']}  close={row['close']}  vol={row['volume']:,}")

get_current_price(AAPL):
{
  "ticker": "AAPL",
  "price": 263.75,
  "change": -0.75,
  "change_pct": -0.28,
  "volume": 38467800,
  "market_cap": 3876577982500.0
}

get_price_history(NVDA, 5d) — 5 rows:
  2026-02-25  close=195.56  vol=250,637,100
  2026-02-26  close=184.89  vol=360,807,900
  2026-02-27  close=177.19  vol=311,636,500
  2026-03-02  close=182.48  vol=209,095,300
  2026-03-03  close=180.05  vol=177,582,000


## Cell 4 — Single-turn Claude query

A single `messages.create()` call that may return `tool_use` blocks.  
We detect them, call the MCP tool, then send the result back to Claude to get the final answer.

This is the minimal agentic pattern — one question, one round of tool use.

In [ ]:
def _mcp_tools_to_anthropic(tools) -> list[dict]:
    """Convert MCP tool definitions to the format Anthropic's API expects."""
    return [
        {
            "name": t.name,
            "description": t.description or "",
            "input_schema": t.inputSchema,
        }
        for t in tools
    ]


# This is the Host that owns overall "user" interaction.  It decides when to talk to Claude and when to invoke
# the MCP client, and how to present results to the user.
# The host is the "brain" orchestrating the conversation and tool use.  
# The MCP client is a protocol adapter the host uses to reacht the server and call tools
async def single_turn_query(question: str) -> str:
    """Ask Claude one question. Resolve one round of tool_use, return final text."""
    anthropic = Anthropic(api_key=ANTHROPIC_API_KEY)

    # sse_client and ClientSession are the MCP client that connects to the MCP server
    # at the MCP_SERVER_URL 
    async with sse_client(MCP_SERVER_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_result = await session.list_tools()
            anthropic_tools = _mcp_tools_to_anthropic(tools_result.tools)

            messages = [{"role": "user", "content": question}]

            # First call — Claude may request tool use
            response = anthropic.messages.create(
                model="claude-sonnet-4-6",
                max_tokens=1024,
                system=SYSTEM_PROMPT,
                tools=anthropic_tools,
                messages=messages,
            )

            if response.stop_reason == "end_turn":
                # Claude answered without needing tools
                return next(b.text for b in response.content if hasattr(b, "text"))

            # Handle tool_use blocks
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []

            for block in response.content:
                if block.type != "tool_use":
                    continue
                args_str = json.dumps(block.input, separators=(",", ":"))
                print(f"  [MCP] → {block.name}({args_str})")

                mcp_result = await session.call_tool(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": mcp_result.content[0].text if mcp_result.content else "{}",
                })

            messages.append({"role": "user", "content": tool_results})

            # Second call — Claude formulates final answer
            final = anthropic.messages.create(
                model="claude-sonnet-4-6",
                max_tokens=1024,
                system=SYSTEM_PROMPT,
                tools=anthropic_tools,
                messages=messages,
            )
            return next(b.text for b in final.content if hasattr(b, "text"))


answer = await single_turn_query("What is NVDA's current price?")
print(f"\nAnswer: {answer}")

## Cell 5 — Multi-turn agentic loop

A reusable `agentic_loop()` that:
- Keeps a running conversation history
- Loops until Claude returns `end_turn` (i.e. until all tool calls are resolved)
- Mirrors the pattern used in the CLI client from Phases 1 and 2

The outer `chat()` function opens one SSE session and runs multiple questions
within it, showing how conversation context accumulates across turns.

In [ ]:
async def agentic_loop(
    question: str,
    conversation: list[dict],
    session: ClientSession,
    anthropic_tools: list[dict],
    anthropic_client: Anthropic,
) -> str:
    """
    Append `question` to the conversation, then loop until Claude stops
    requesting tools. Returns the final text answer.
    """
    conversation.append({"role": "user", "content": question})

    while True:
        response = anthropic_client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=4096,
            system=SYSTEM_PROMPT,
            tools=anthropic_tools,
            messages=conversation,
        )

        if response.stop_reason == "end_turn":
            text = next(
                (b.text for b in response.content if hasattr(b, "text")),
                "(no text)",
            )
            conversation.append({"role": "assistant", "content": response.content})
            return text

        if response.stop_reason == "tool_use":
            conversation.append({"role": "assistant", "content": response.content})
            tool_results = []

            for block in response.content:
                if block.type != "tool_use":
                    continue
                args_str = json.dumps(block.input, separators=(",", ":"))
                print(f"    [MCP] → {block.name}({args_str})")

                mcp_result = await session.call_tool(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": mcp_result.content[0].text if mcp_result.content else "{}",
                })

            conversation.append({"role": "user", "content": tool_results})
            # Loop — let Claude process tool results and decide if more calls needed.
        else:
            break

    return "(unexpected stop)"


async def chat(questions: list[str]) -> None:
    """Open one MCP session and run multiple questions, preserving conversation context."""
    anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)
    conversation: list[dict] = []

    async with sse_client(MCP_SERVER_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_result = await session.list_tools()
            anthropic_tools = _mcp_tools_to_anthropic(tools_result.tools)

            for i, question in enumerate(questions, 1):
                print(f"Q{i}: {question}")
                answer = await agentic_loop(
                    question, conversation, session, anthropic_tools, anthropic_client
                )
                print(f"A{i}: {answer}\n")


await chat([
    "What is MSFT's current price and how has it changed today?",
    "How does that compare to its 52-week range?",   # follow-up — uses conversation context
    "Which of AAPL, MSFT, and NVDA has the highest P/E ratio?",
])